In [1]:
!pip install -q sentence-transformers scikit-learn

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
documents = [
    "Python is a popular programming language used for data science and machine learning.",
    "Python supports web development, automation, and scripting.",
    "Machine learning allows computers to learn patterns from data.",
    "Pandas is a Python library used for data manipulation and analysis.",
    "NumPy provides multidimensional arrays and numerical computing capabilities.",
    "PyTorch is a deep learning framework used to build and train neural networks.",
    "Computer vision enables computers to understand images and videos.",
    "Natural language processing enables computers to understand and generate human language."
]

for i, doc in enumerate(documents):
    print(f"{i}: {doc}")

0: Python is a popular programming language used for data science and machine learning.
1: Python supports web development, automation, and scripting.
2: Machine learning allows computers to learn patterns from data.
3: Pandas is a Python library used for data manipulation and analysis.
4: NumPy provides multidimensional arrays and numerical computing capabilities.
5: PyTorch is a deep learning framework used to build and train neural networks.
6: Computer vision enables computers to understand images and videos.
7: Natural language processing enables computers to understand and generate human language.


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (8, 384)


Create Retrieval Function

In [5]:
def retrieve(query, k=5):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Calculate similarity
    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Sort from highest to lowest
    ranked_indices = np.argsort(scores)[::-1]

    # Select top K
    top_indices = ranked_indices[:k]

    return top_indices, scores[top_indices]

Test Retrieval

In [6]:
query = "What is Pandas used for?"

indices, scores = retrieve(query, k=5)

print("Query:", query)
print("\nTop 5 Results:\n")

for rank, (index, score) in enumerate(
    zip(indices, scores),
    start=1
):
    print(f"Rank {rank}")
    print(f"Document ID: {index}")
    print(f"Similarity: {score:.4f}")
    print(f"Text: {documents[index]}")
    print("-" * 70)

Query: What is Pandas used for?

Top 5 Results:

Rank 1
Document ID: 3
Similarity: 0.6493
Text: Pandas is a Python library used for data manipulation and analysis.
----------------------------------------------------------------------
Rank 2
Document ID: 0
Similarity: 0.3279
Text: Python is a popular programming language used for data science and machine learning.
----------------------------------------------------------------------
Rank 3
Document ID: 1
Similarity: 0.2433
Text: Python supports web development, automation, and scripting.
----------------------------------------------------------------------
Rank 4
Document ID: 4
Similarity: 0.2413
Text: NumPy provides multidimensional arrays and numerical computing capabilities.
----------------------------------------------------------------------
Rank 5
Document ID: 5
Similarity: 0.2299
Text: PyTorch is a deep learning framework used to build and train neural networks.
----------------------------------------------------------------

Create Ground Truth

In [7]:
query = "What is Pandas used for?"

relevant_documents = [3]

Precision@K

In [8]:
def precision_at_k(retrieved_ids, relevant_ids, k):

    retrieved_ids = retrieved_ids[:k]

    relevant_count = sum(
        1
        for doc_id in retrieved_ids
        if doc_id in relevant_ids
    )

    return relevant_count / k

In [9]:
retrieved_ids = indices
relevant_ids = [3]

score = precision_at_k(
    retrieved_ids,
    relevant_ids,
    k=5
)

print("Precision@5:", score)
print("Precision@5 (%):", score * 100)

Precision@5: 0.2
Precision@5 (%): 20.0


Test Precision@1, @3 and @5

In [10]:
for k in [1, 3, 5]:

    score = precision_at_k(
        retrieved_ids,
        relevant_ids,
        k
    )

    print(
        f"Precision@{k}: "
        f"{score:.2f} "
        f"({score * 100:.1f}%)"
    )

Precision@1: 1.00 (100.0%)
Precision@3: 0.33 (33.3%)
Precision@5: 0.20 (20.0%)


Recall@K

In [11]:
def recall_at_k(retrieved_ids, relevant_ids, k):

    retrieved_ids = retrieved_ids[:k]

    relevant_count = sum(
        1
        for doc_id in retrieved_ids
        if doc_id in relevant_ids
    )

    return relevant_count / len(relevant_ids)

In [13]:
for k in [1, 3, 5]:

    score = recall_at_k(
        retrieved_ids,
        relevant_ids,
        k
    )

    print(
        f"Recall@{k}: "
        f"{score:.2f} "
        f"({score * 100:.1f}%)"
    )

Recall@1: 1.00 (100.0%)
Recall@3: 1.00 (100.0%)
Recall@5: 1.00 (100.0%)


Hit Rate@K

In [14]:
def hit_rate_at_k(
    retrieved_ids,
    relevant_ids,
    k
):

    retrieved_ids = retrieved_ids[:k]

    for doc_id in retrieved_ids:

        if doc_id in relevant_ids:
            return 1

    return 0

In [15]:
for k in [1, 3, 5]:

    score = hit_rate_at_k(
        retrieved_ids,
        relevant_ids,
        k
    )

    print(f"Hit Rate@{k}: {score}")

Hit Rate@1: 1
Hit Rate@3: 1
Hit Rate@5: 1


MRR

Now let's calculate Mean Reciprocal Rank for one query.

MRR focuses on the position of the first relevant result.

In [16]:
def reciprocal_rank(
    retrieved_ids,
    relevant_ids
):

    for rank, doc_id in enumerate(
        retrieved_ids,
        start=1
    ):

        if doc_id in relevant_ids:
            return 1 / rank

    return 0

In [17]:
rr = reciprocal_rank(
    retrieved_ids,
    relevant_ids
)

print("Reciprocal Rank:", rr)

Reciprocal Rank: 1.0


In [18]:
evaluation_dataset = [
    {
        "query": "What is Python?",
        "relevant_ids": [0, 1]
    },
    {
        "query": "What is machine learning?",
        "relevant_ids": [2]
    },
    {
        "query": "What is Pandas used for?",
        "relevant_ids": [3]
    },
    {
        "query": "What is NumPy?",
        "relevant_ids": [4]
    },
    {
        "query": "What is PyTorch?",
        "relevant_ids": [5]
    },
    {
        "query": "What is computer vision?",
        "relevant_ids": [6]
    },
    {
        "query": "What is NLP?",
        "relevant_ids": [7]
    }
]

In [19]:
def evaluate_retrieval(
    evaluation_dataset,
    k=5
):

    precision_scores = []
    recall_scores = []
    hit_scores = []
    rr_scores = []

    for item in evaluation_dataset:

        query = item["query"]
        relevant_ids = item["relevant_ids"]

        retrieved_ids, scores = retrieve(
            query,
            k=k
        )

        precision = precision_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )

        recall = recall_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )

        hit = hit_rate_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )

        rr = reciprocal_rank(
            retrieved_ids,
            relevant_ids
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit)
        rr_scores.append(rr)

        print(f"\nQuery: {query}")
        print(f"Retrieved IDs: {list(retrieved_ids)}")
        print(f"Relevant IDs: {relevant_ids}")
        print(f"Precision@{k}: {precision:.2f}")
        print(f"Recall@{k}: {recall:.2f}")
        print(f"Hit Rate@{k}: {hit}")
        print(f"RR: {rr:.2f}")

    print("\n" + "=" * 60)
    print("OVERALL RESULTS")
    print("=" * 60)

    print(
        f"Mean Precision@{k}: "
        f"{np.mean(precision_scores):.3f}"
    )

    print(
        f"Mean Recall@{k}: "
        f"{np.mean(recall_scores):.3f}"
    )

    print(
        f"Hit Rate@{k}: "
        f"{np.mean(hit_scores):.3f}"
    )

    print(
        f"MRR: "
        f"{np.mean(rr_scores):.3f}"
    )

In [20]:
for k in [1, 3, 5]:

    print("\n")
    print("=" * 60)
    print(f"EVALUATION FOR K = {k}")
    print("=" * 60)

    evaluate_retrieval(
        evaluation_dataset,
        k=k
    )



EVALUATION FOR K = 1

Query: What is Python?
Retrieved IDs: [np.int64(0)]
Relevant IDs: [0, 1]
Precision@1: 1.00
Recall@1: 0.50
Hit Rate@1: 1
RR: 1.00

Query: What is machine learning?
Retrieved IDs: [np.int64(2)]
Relevant IDs: [2]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

Query: What is Pandas used for?
Retrieved IDs: [np.int64(3)]
Relevant IDs: [3]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

Query: What is NumPy?
Retrieved IDs: [np.int64(4)]
Relevant IDs: [4]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

Query: What is PyTorch?
Retrieved IDs: [np.int64(5)]
Relevant IDs: [5]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

Query: What is computer vision?
Retrieved IDs: [np.int64(6)]
Relevant IDs: [6]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

Query: What is NLP?
Retrieved IDs: [np.int64(7)]
Relevant IDs: [7]
Precision@1: 1.00
Recall@1: 1.00
Hit Rate@1: 1
RR: 1.00

OVERALL RESULTS
Mean Precision@1: 1.000
Mean Recall@1: 0.92